# Stage 2C: SparseGPT Pruning with Cross-Play Calibration

**Goal:** Apply SparseGPT unstructured pruning to the **Temporal Transformer only**
(depth transformer is left untouched) using cross-play conversations as calibration data.

### Why Cross-Play Calibration?

Moshi is full-duplex: its outputs at time *t* become inputs at *t+1*. Cross-play
conversations (two teacher models talking) capture this feedback loop naturally,
producing calibration data that perfectly matches the real inference distribution.
This gives SparseGPT more accurate Hessian estimates than static data.

### Pruning Strategy
| Component | Action |
|-----------|--------|
| **Temporal Transformer** | SparseGPT at configurable sparsity (default 30%) |
| **Depth Transformer** | Untouched — directly governs audio codebook quality |
| **Embeddings / LM Head** | Untouched |
| **Mimi Codec** | Untouched |

### Requirements
- **GPU:** L4 (24 GB) or better
- **Data:** Cross-play conversations from `04a_cross_play_generation.ipynb`
  (at least `xplay_A_00` batch with 48+ conversations)

## Cell 1: Session Startup

In [ ]:
# === SESSION STARTUP ===
from google.colab import drive, userdata
drive.mount('/content/drive')

import subprocess, sys, os

# Fetch GitHub token from Colab Secrets
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    print("Warning: GITHUB_TOKEN not found in Colab Secrets.")
    GITHUB_TOKEN = ""

REPO_OWNER = "RidwanHR5"
REPO_NAME = "moshilite"

# Construct URL with auth token
if GITHUB_TOKEN:
    REPO = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
else:
    REPO = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

REPO_DIR = "/content/moshilite"

# Clone or pull latest code
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

# Install as an editable package
subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR, "-q"], check=True)
print("moshilite package installed successfully!")

## Cell 2: Configuration

**Change the sparsity target here.** Everything else auto-computes from this.

In [ ]:
import torch, gc, os, json, time
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ============================================================
#  ⚙️  CONFIGURATION — CHANGE THESE VALUES
# ============================================================

# Sparsity target: fraction of temporal transformer weights to zero out.
# Recommended values: 0.20, 0.25, 0.30
SPARSITY = 0.30

# SparseGPT hyperparameters
BLOCK_SIZE = 128         # Column block size for blocked OBS updates
PERCDAMP = 0.01          # Dampening factor for Hessian diagonal

# Calibration data config
CALIBRATION_SEQ_LEN = 500   # Steps per chunk (500 ≈ 40s @ 12.5 Hz)
CALIBRATION_BATCH_SIZE = 4  # Batch size for calibration forward passes
# Each 5-min conversation (3750 steps) is chunked into ~7 segments of 500 steps.
# 96 cross-play files → ~672 calibration samples. Uses 100% of the data.

# ============================================================
#  AUTO-COMPUTED — DO NOT EDIT BELOW
# ============================================================

SPARSITY_PCT = int(SPARSITY * 100)
VARIANT_NAME = f"sparsegpt_xplay_{SPARSITY_PCT}pct"
EXPERIMENT_ID = f"prune{SPARSITY_PCT}_xplay"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REPO_DIR = "/content/moshilite"
WEIGHTS_DIR = "/content/drive/MyDrive/moshilite/upstream_weights/moshiko"

CHECKPOINT_DIR = f"/content/drive/MyDrive/moshilite/checkpoints/{EXPERIMENT_ID}"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

XPLAY_DATA_DIR = "/content/drive/MyDrive/moshilite/self_play_data/conversations"

print("=" * 70)
print(f"  SparseGPT Pruning with Cross-Play Calibration")
print("=" * 70)
print(f"  Sparsity target:     {SPARSITY:.0%}")
print(f"  Variant name:        {VARIANT_NAME}")
print(f"  Block size:          {BLOCK_SIZE}")
print(f"  Percdamp:            {PERCDAMP}")
print(f"  Chunk length:        {CALIBRATION_SEQ_LEN} steps ({CALIBRATION_SEQ_LEN/12.5:.0f}s)")
print(f"  Checkpoint dir:      {CHECKPOINT_DIR}")
print(f"  Cross-play data:     {XPLAY_DATA_DIR}")
print(f"  Device:              {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU:                 {torch.cuda.get_device_name()}")
    print(f"  VRAM:                {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Cell 3: Helpers

In [ ]:
from moshilite.analysis.moshi_model import load_moshi_for_analysis, get_model_info
from moshilite.pruning.unstructured_pruning import (
    prune_sparsegpt,
    get_model_sparsity,
)


def load_fresh_model():
    """Load a fresh Moshi model in BF16. Call before pruning."""
    print("Loading Moshi in BF16...")
    model = load_moshi_for_analysis(
        precision="bf16", device=DEVICE, weights_dir=WEIGHTS_DIR
    )
    info = get_model_info(model)
    print(f"Loaded: {info['total_params_b']:.3f}B params")
    print(f"  n_layers={info.get('n_layers', '?')}, "
          f"n_heads={info.get('n_heads', '?')}, "
          f"d_model={info.get('d_model', '?')}")
    if DEVICE == "cuda":
        print(f"  VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")
    return model


def cleanup():
    """Force garbage collection and free GPU cache."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB")


def save_pruned_model(model, variant_name, result, prune_time):
    """Save pruned model checkpoint with metadata."""
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"{variant_name}.pt")
    sparsity_stats = get_model_sparsity(model)

    torch.save({
        "model_state_dict": model.state_dict(),
        "variant_name": variant_name,
        "pruning_method": "sparsegpt",
        "pruning_type": "unstructured",
        "calibration_source": "cross_play",
        "calibration_convos": N_CALIBRATION_CONVOS,
        "calibration_seq_len": CALIBRATION_SEQ_LEN,
        "target_sparsity": SPARSITY,
        "actual_sparsity": result.actual_sparsity,
        "param_count": sum(p.numel() for p in model.parameters()),
        "param_billions": round(sum(p.numel() for p in model.parameters()) / 1e9, 3),
        "nonzero_params": sparsity_stats["nonzero_params"],
        "nonzero_param_billions": round(sparsity_stats["nonzero_params"] / 1e9, 3),
        "zeroed_params": result.zeroed_params,
        "block_size": BLOCK_SIZE,
        "percdamp": PERCDAMP,
        "pruning_time_s": prune_time,
        "experiment_id": EXPERIMENT_ID,
        "scope": "temporal_only",  # depth transformer untouched
    }, ckpt_path)

    print(f"\n💾 Saved: {ckpt_path}")
    print(f"   File size:       {os.path.getsize(ckpt_path)/1e9:.2f} GB")
    print(f"   Total params:    {sum(p.numel() for p in model.parameters())/1e9:.3f}B")
    print(f"   Non-zero params: {sparsity_stats['nonzero_params']/1e9:.3f}B")
    print(f"   Actual sparsity: {result.actual_sparsity:.2%}")
    print(f"   Pruning time:    {prune_time:.1f}s ({prune_time/60:.1f} min)")
    return ckpt_path


print("Helpers ready.")

## Cell 4: Load Cross-Play Calibration Data

Loads `.npz` files from the cross-play batch (`xplay_A_00/` etc.) and
constructs token tensors in the format expected by Moshi's forward pass:
`[B, 17, T]` where channels are `[text, model_cb0..7, user_cb0..7]`.

**Priority:** Prefers cross-play data (`*_modelA.npz`, `*_modelB.npz`).
Falls back to self-play data if no cross-play is available.

In [ ]:
import numpy as np
from pathlib import Path

def load_cross_play_calibration(
    data_dir: str,
    seq_len: int = 500,
    batch_size: int = 4,
) -> list[torch.Tensor]:
    """Load cross-play conversations as calibration data for SparseGPT.

    Each conversation (3750 steps) is chunked into multiple segments of
    seq_len steps, so 100% of the data is used for Hessian estimation.
    """
    N_CHANNELS = 17
    conv_dir = Path(data_dir)

    # Find cross-play .npz files (prefer xplay batches)
    xplay_files = sorted(conv_dir.rglob("*_model[AB].npz"))
    selfplay_files = sorted(
        f for f in conv_dir.rglob("conv_*.npz")
        if "_model" not in f.stem and "rejected" not in str(f)
    )

    if xplay_files:
        npz_files = xplay_files
        source = "cross-play"
    elif selfplay_files:
        npz_files = selfplay_files
        source = "self-play (fallback)"
    else:
        raise FileNotFoundError(
            f"No conversation .npz files found in {data_dir}\n"
            f"Run 04a_cross_play_generation.ipynb first."
        )

    print(f"Found {len(npz_files)} {source} conversations")

    all_chunks = []
    for npz_path in npz_files:
        data = np.load(str(npz_path))
        text_tokens = data["text_tokens"]
        audio_tokens = data["audio_tokens"]
        user_audio = data.get("user_audio_tokens")

        T_total = len(text_tokens)

        # Chunk this conversation into segments of seq_len
        for start in range(0, T_total - seq_len + 1, seq_len):
            end = start + seq_len
            tokens = np.zeros((N_CHANNELS, seq_len), dtype=np.int64)
            tokens[0, :] = text_tokens[start:end]

            n_model = min(audio_tokens.shape[0], 8)
            tokens[1:1 + n_model, :] = audio_tokens[:n_model, start:end]

            if user_audio is not None:
                n_user = min(user_audio.shape[0], 8)
                tokens[9:9 + n_user, :] = user_audio[:n_user, start:end]

            all_chunks.append(tokens)

    # Group into batches
    batches = []
    for i in range(0, len(all_chunks), batch_size):
        batch = all_chunks[i:i + batch_size]
        while len(batch) < batch_size:
            batch.append(batch[-1])
        tensor = torch.tensor(np.stack(batch), dtype=torch.long)
        batches.append(tensor)

    print(f"Chunked into {len(all_chunks)} segments → {len(batches)} calibration batches")
    print(f"  Shape: [{batch_size}, {N_CHANNELS}, {seq_len}]")
    return batches


# ── Load calibration data ──
calibration_data = load_cross_play_calibration(
    data_dir=XPLAY_DATA_DIR,
    seq_len=CALIBRATION_SEQ_LEN,
    batch_size=CALIBRATION_BATCH_SIZE,
)

sample = calibration_data[0]
print(f"\n✅ Calibration data ready")
print(f"   Sample shape: {sample.shape}")
print(f"   Text tokens range: [{sample[0, 0].min()}, {sample[0, 0].max()}]")
print(f"   Audio tokens range: [{sample[0, 1:9].min()}, {sample[0, 1:9].max()}]")


---
## Cell 5: Run SparseGPT Pruning

This is the main pruning cell. It:
1. Loads a fresh Moshi model (BF16)
2. Runs calibration forward passes to accumulate Hessians (H = X^T @ X)
3. For each Linear layer in the Temporal Transformer, prunes with
   Hessian-based weight reconstruction
4. Saves the pruned checkpoint

**Expected time:** ~20 min calibration + ~5 min pruning on L4.

⚠️ The Depth Transformer is **not touched** — `_get_prunable_linears()` only
targets `model.transformer` (the Temporal Transformer).

In [ ]:
print("=" * 70)
print(f"  SparseGPT PRUNING ({SPARSITY:.0%}) — Cross-Play Calibration")
print(f"  Temporal Transformer only. Depth Transformer untouched.")
print("=" * 70)
print(f"  Block size: {BLOCK_SIZE}, Percdamp: {PERCDAMP}")
print(f"  Calibration: {len(calibration_data)} batches × "
      f"[{CALIBRATION_BATCH_SIZE}, 17, {CALIBRATION_SEQ_LEN}]")
print()

# 1. Load fresh model
model = load_fresh_model()

# 2. Run SparseGPT (calibration + Hessian + prune + reconstruct)
print(f"\n🔧 Starting SparseGPT at {SPARSITY:.0%} sparsity...")
print(f"   This will take ~20-30 minutes. Be patient.\n")

t0 = time.time()
result = prune_sparsegpt(
    model,
    calibration_data=calibration_data,
    sparsity=SPARSITY,
    block_size=BLOCK_SIZE,
    percdamp=PERCDAMP,
    device=DEVICE,
)
prune_time = time.time() - t0

# 3. Report results
print(f"\n{'=' * 70}")
print(f"  PRUNING COMPLETE")
print(f"{'=' * 70}")
print(f"  Method:          SparseGPT")
print(f"  Target sparsity: {result.target_sparsity:.1%}")
print(f"  Actual sparsity: {result.actual_sparsity:.1%}")
print(f"  Total params:    {result.total_params:,}")
print(f"  Zeroed params:   {result.zeroed_params:,}")
print(f"  Layers pruned:   {result.n_layers_pruned}")
print(f"  Time:            {prune_time:.1f}s ({prune_time/60:.1f} min)")

# 4. Save checkpoint
ckpt_path = save_pruned_model(model, VARIANT_NAME, result, prune_time)

## Cell 6: Per-Layer Sparsity Analysis

Inspect which layers were pruned more/less aggressively.
Useful to understand if certain layers are more robust to pruning.

In [ ]:
# Per-layer sparsity breakdown
print(f"\n{'Layer':<55} {'Sparsity':>10}")
print("-" * 67)

layer_groups = {}  # group by layer index
for name, sparsity in sorted(result.per_layer_sparsity.items()):
    # Extract layer index from name like "transformer.layers.0.self_attn.in_projs.0"
    parts = name.split(".")
    if "layers" in parts:
        layer_idx = int(parts[parts.index("layers") + 1])
    else:
        layer_idx = -1
    layer_groups.setdefault(layer_idx, []).append((name, sparsity))

for layer_idx in sorted(layer_groups.keys()):
    entries = layer_groups[layer_idx]
    avg_sparsity = sum(s for _, s in entries) / len(entries)
    print(f"\n  Layer {layer_idx} (avg: {avg_sparsity:.1%}):")
    for name, sparsity in entries:
        short_name = name.replace("transformer.layers.", "L")
        bar = "█" * int(sparsity * 30) + "░" * (30 - int(sparsity * 30))
        print(f"    {short_name:<50} {sparsity:6.1%}  {bar}")

# Summary statistics
all_sparsities = list(result.per_layer_sparsity.values())
print(f"\n{'=' * 67}")
print(f"  Min layer sparsity:  {min(all_sparsities):.1%}")
print(f"  Max layer sparsity:  {max(all_sparsities):.1%}")
print(f"  Mean layer sparsity: {sum(all_sparsities)/len(all_sparsities):.1%}")
print(f"  Std layer sparsity:  {torch.tensor(all_sparsities).std():.3f}")

## Cell 7: Verification

Load the saved checkpoint and independently verify sparsity.
Also confirms the depth transformer was left untouched.

In [ ]:
# Free the in-memory model first
del model
cleanup()

print("=" * 70)
print("  VERIFICATION")
print("=" * 70)

# Load checkpoint
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
sd = ckpt["model_state_dict"]

# 1. Verify temporal transformer sparsity
temporal_total, temporal_zero = 0, 0
for key, tensor in sd.items():
    if "transformer" in key and "weight" in key and tensor.dim() == 2:
        temporal_total += tensor.numel()
        temporal_zero += (tensor == 0).sum().item()

temporal_sparsity = temporal_zero / temporal_total if temporal_total > 0 else 0
saved_sparsity = ckpt.get("actual_sparsity", -1)
match = abs(temporal_sparsity - saved_sparsity) < 0.01

print(f"\n  Temporal Transformer:")
print(f"    Verified sparsity: {temporal_sparsity:.2%}")
print(f"    Saved sparsity:    {saved_sparsity:.2%}")
print(f"    Match:             {'✅ PASS' if match else '❌ MISMATCH'}")

# 2. Verify depth transformer is untouched (zero sparsity)
depth_total, depth_zero = 0, 0
for key, tensor in sd.items():
    if "depformer" in key.lower() and "weight" in key and tensor.dim() == 2:
        depth_total += tensor.numel()
        depth_zero += (tensor == 0).sum().item()

if depth_total > 0:
    depth_sparsity = depth_zero / depth_total
    depth_ok = depth_sparsity < 0.01  # Should be ~0%
    print(f"\n  Depth Transformer:")
    print(f"    Sparsity:          {depth_sparsity:.2%}")
    print(f"    Untouched:         {'✅ YES' if depth_ok else '❌ NO — something went wrong!'}")
else:
    # Try alternative naming
    depth_keys = [k for k in sd.keys() if "dep" in k.lower()]
    if depth_keys:
        depth_total = sum(sd[k].numel() for k in depth_keys if sd[k].dim() == 2)
        depth_zero = sum((sd[k] == 0).sum().item() for k in depth_keys if sd[k].dim() == 2)
        depth_sparsity = depth_zero / depth_total if depth_total > 0 else 0
        print(f"\n  Depth Transformer ({len(depth_keys)} keys):")
        print(f"    Sparsity:          {depth_sparsity:.2%}")
        print(f"    Untouched:         {'✅ YES' if depth_sparsity < 0.01 else '❌ NO'}")
    else:
        print(f"\n  Depth Transformer: (no 'dep*' keys found in state_dict)")

# 3. File size
print(f"\n  Checkpoint:")
print(f"    Path: {ckpt_path}")
print(f"    Size: {os.path.getsize(ckpt_path)/1e9:.2f} GB")
print(f"    Params (total):    {ckpt['param_billions']}B")
print(f"    Params (non-zero): {ckpt['nonzero_param_billions']}B")
print(f"    Calibration:       {ckpt.get('calibration_source', '?')} "
      f"({ckpt.get('calibration_convos', '?')} convos)")
print(f"    Scope:             {ckpt.get('scope', '?')}")

del ckpt, sd
gc.collect()
print(f"\n✅ Verification complete.")

## Cell 8: Comparison with Previous Variants (Optional)

Shows all pruned variants in the checkpoint directories for comparison.

In [ ]:
print("\n" + "=" * 90)
print("  ALL PRUNED VARIANTS (comparison)")
print("=" * 90)

ckpt_dirs = [
    "/content/drive/MyDrive/moshilite/checkpoints/prune30_v1",     # Previous experiments
    CHECKPOINT_DIR,                                                  # This experiment
]

print(f"\n{'Variant':<35} {'Calibration':<14} {'Sparsity':>10} {'Non-Zero':>12} {'Size (GB)':>10}")
print("-" * 83)

for ckpt_dir in ckpt_dirs:
    if not os.path.exists(ckpt_dir):
        continue
    for f in sorted(os.listdir(ckpt_dir)):
        if not f.endswith(".pt"):
            continue
        path = os.path.join(ckpt_dir, f)
        size_gb = os.path.getsize(path) / 1e9

        try:
            ckpt = torch.load(path, map_location="cpu", weights_only=False)
            name = ckpt.get("variant_name", f.replace(".pt", ""))
            calib = ckpt.get("calibration_source", "self-play")
            sparsity = ckpt.get("actual_sparsity", 0)
            nz = ckpt.get("nonzero_param_billions", "?")
            print(f"  {name:<33} {calib:<14} {sparsity:>8.1%}   {nz:>10}B   {size_gb:>8.2f}")
            del ckpt
        except Exception as e:
            print(f"  {f.replace('.pt', ''):<33} {'?':<14} {'?':>10} {'?':>12} {size_gb:>8.2f}")

        gc.collect()

print(f"\n✅ Cross-play calibrated variant: {VARIANT_NAME}")
print(f"   Next step: Run KD training (04b_student_kd_training.ipynb) with this checkpoint.")